<a href="https://colab.research.google.com/github/amit-sahu-a11y/ML_projects_for_practice/blob/main/parse.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

To install a Python library in Colab, you can use `pip install` with an exclamation mark prefix within a code cell. For example, to install the `requests` library, you would run:

In [38]:
!pip install requests

After installation, you can import and use the library in subsequent cells:

In [39]:
import requests

response = requests.get('https://www.google.com')
print(response.status_code)
print(response)


200
<Response [200]>


In [40]:
!pip install beautifulsoup4

Now that `beautifulsoup4` is installed, you can import `BeautifulSoup` and use it to parse the `response.text` from your previous `requests.get` call:

In [41]:
from bs4 import BeautifulSoup

# Assuming 'response' variable still holds the requests.get() result
soup = BeautifulSoup(response.text, 'html.parser')

# For example, let's print the title of the page
print(f"Page title: {soup.title.string}")

# Or find all links
# for link in soup.find_all('a'):
#     print(link.get('href'))

Page title: Google


Let's change our target URL to a Wikipedia page with a table to demonstrate how to scrape tabular data. We'll use the 'List of Python libraries' page as an example.

Now that we have the HTML content of a page with tables, we can find and extract data from a specific table. Wikipedia pages often have multiple tables, so we might need to identify the correct one. I'll look for the first table on the page and extract its header and row data.

It appears the cells that scraped the table data were deleted. Let's recreate them to ensure `df_table` is available.

In [42]:
# Re-fetching the page and parsing it to create soup_tables
import requests
from bs4 import BeautifulSoup

# Trying a more relevant URL for a list of Python software.
url_with_tables = 'https://en.wikipedia.org/wiki/List_of_Python_software'
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}
response_tables = requests.get(url_with_tables, headers=headers)
response_tables.raise_for_status() # Raise an exception for bad status codes

soup_tables = BeautifulSoup(response_tables.text, 'html.parser')
print(f"Successfully fetched and parsed: {url_with_tables}")

Successfully fetched and parsed: https://en.wikipedia.org/wiki/List_of_Python_software


In [50]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from io import StringIO

# Correcting the URL to a valid Wikipedia page that contains comparison tables.
url_with_tables = 'https://en.wikipedia.org/wiki/Comparison_of_web_frameworks'
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}
response_tables = requests.get(url_with_tables, headers=headers)
response_tables.raise_for_status() # Raise an exception for bad status codes

print(f"Successfully fetched: {url_with_tables}")

# --- New diagnostic: Check for raw <table> string ---
if '<table' in response_tables.text.lower():
    print("Raw '<table' string found in response_tables.text. Proceeding with BeautifulSoup parsing.")
else:
    print("Raw '<table' string NOT found in response_tables.text. The page content might not contain tables or be incomplete.")
    print("This indicates a potential issue with content retrieval rather than parsing.")
    # If tables are not found as raw strings, there's no point in further BeautifulSoup processing for tables.
    # We'll still try, but expect it to fail.

soup = BeautifulSoup(response_tables.text, 'html.parser')
df_table = pd.DataFrame() # Initialize df_table as an empty DataFrame

# Find all <table> tags on the page
html_tables = soup.find_all('table')

if html_tables:
    print(f"Found {len(html_tables)} <table> elements on the page. Attempting to parse them.")
    table_found = False
    for i, table_tag in enumerate(html_tables):
        try:
            # Try to parse each table tag individually
            dfs = pd.read_html(StringIO(str(table_tag)))
            if dfs:
                df_table = dfs[0]
                print(f"DataFrame 'df_table' created successfully from table {i+1} using pd.read_html.")
                display(df_table.head())
                table_found = True
                break # Stop after finding and parsing the first successful table
        except Exception as e:
            print(f"Could not parse table {i+1} with pd.read_html: {e}")

    if not table_found:
        print("No parsable tables found among the <table> elements.")
else:
    print("No <table> elements found on the page using BeautifulSoup.")

Successfully fetched: https://en.wikipedia.org/wiki/Comparison_of_web_frameworks

--- First 5000 characters of raw HTML content ---
<!DOCTYPE html>
<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-content-enabled vector-feature-custom-font-size-clientpref-1 vector-feature-appearance-pinned-clientpref-1 skin-theme-clientpref-day vector-sticky-header-enabled vector-toc-not-available skin-thumbsize-clientpref-standard" lang="en" dir="ltr">
<head>
<meta charset="UTF-8">
<title>Comparison of web frameworks - Wikipedia</title>
<script>(function(){var className="client-js vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-mai

In [44]:
# Display the first few rows of df_table to verify the data
display(df_table.head())

""


In [45]:
import pandas as pd

# Ensure df_table exists from previous steps, otherwise replace with your DataFrame
# For demonstration, let's assume df_table is already available and populated.
# If it's not, you'd need to re-run the scraping code or load sample data.

# Save the DataFrame to a CSV file
output_csv_filename = 'wikipedia_python_libraries_table.csv'
df_table.to_csv(output_csv_filename, index=False) # index=False prevents writing the DataFrame index as a column

print(f"Data successfully saved to {output_csv_filename}")

# Optionally, display the first few rows from the saved CSV to verify
# df_read_from_csv = pd.read_csv(output_csv_filename)
# display(df_read_from_csv.head())

Data successfully saved to wikipedia_python_libraries_table.csv


In [46]:
from bs4 import BeautifulSoup

# Assuming soup_tables object is available from previous steps
# If not, ensure d3ac2a3c is run to create soup_tables

# Get all elements that have a 'class' attribute
all_elements_with_class = soup_tables.find_all(class_=True)

unique_classes = set()
for element in all_elements_with_class:
    if 'class' in element.attrs:
        for cls in element['class']:
            unique_classes.add(cls)

print("Unique CSS classes found in the HTML:")
for cls in sorted(list(unique_classes)):
    print(f"- {cls}")

Unique CSS classes found in the HTML:
- action-view
- after-portlet
- after-portlet-lang
- anonymous-show
- catlinks
- cdx-button
- cdx-button--action-progressive
- cdx-button--fake-button
- cdx-button--fake-button--enabled
- cdx-button--icon-only
- cdx-button--size-large
- cdx-button--weight-quiet
- cdx-button__icon
- cdx-search-input
- cdx-search-input--has-end-button
- cdx-search-input__end-button
- cdx-search-input__input-wrapper
- cdx-text-input
- cdx-text-input--has-start-icon
- cdx-text-input__icon
- cdx-text-input__input
- cdx-text-input__start-icon
- cdx-typeahead-search
- cdx-typeahead-search--auto-expand-width
- cdx-typeahead-search--show-thumbnail
- client-nojs
- dmbox
- dmbox-body
- dmbox-setindex
- emptyPortlet
- external
- extiw
- firstHeading
- ltr
- mediawiki
- metadata
- mw-aria-live-region
- mw-body
- mw-body-content
- mw-body-header
- mw-content-container
- mw-content-ltr
- mw-editable
- mw-empty-elt
- mw-file-description
- mw-file-element
- mw-first-heading
- mw-fo